In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"ardraappu","key":"896d13d76a5fa6f2fdc4e2033e995a28"}'}

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!pip install kaggle
!kaggle datasets download -d uciml/sms-spam-collection-dataset
!unzip sms-spam-collection-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset
License(s): unknown
100% 211k/211k [00:00<00:00, 86.3MB/s]

Archive:  sms-spam-collection-dataset.zip
  inflating: spam.csv                


In [3]:
import pandas as pd
df = pd.read_csv('spam.csv', encoding='latin-1')
print(df.head(10))
print(df.columns)
print(df.shape)

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   
5  spam  FreeMsg Hey there darling it's been 3 week's n...        NaN   
6   ham  Even my brother is not like to speak with me. ...        NaN   
7   ham  As per your request 'Melle Melle (Oru Minnamin...        NaN   
8  spam  WINNER!! As a valued network customer you have...        NaN   
9  spam  Had your mobile 11 months or more? U R entitle...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  
5        NaN        NaN  
6        NaN  

In [4]:
df=df[['v1','v2']]
df.columns=['label','message']
print(df.head(10))

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
5  spam  FreeMsg Hey there darling it's been 3 week's n...
6   ham  Even my brother is not like to speak with me. ...
7   ham  As per your request 'Melle Melle (Oru Minnamin...
8  spam  WINNER!! As a valued network customer you have...
9  spam  Had your mobile 11 months or more? U R entitle...


In [5]:
print(df.shape)
print(df['label'].value_counts())  # see how many spam vs ham

(5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64


In [6]:
import string

def clean_text(text):
    text = text.lower()                                          # make everything lowercase
    text = ''.join([c for c in text if c not in string.punctuation])  # remove punctuation like . , ! ?
    return text

df['clean_message'] = df['message'].apply(clean_text)

# check before vs after on a couple of examples
for i in [2, 8]:
    print('BEFORE:', df['message'][i])
    print('AFTER :', df['clean_message'][i])
    print()

BEFORE: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
AFTER : free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005 text fa to 87121 to receive entry questionstd txt ratetcs apply 08452810075over18s

BEFORE: WINNER!! As a valued network customer you have been selected to receivea å£900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.
AFTER : winner as a valued network customer you have been selected to receivea å£900 prize reward to claim call 09061701461 claim code kl341 valid 12 hours only



In [7]:
from sklearn.model_selection import train_test_split

# balance: take all spam, and an equal random number of ham
spam = df[df['label'] == 'spam']
ham = df[df['label'] == 'ham'].sample(n=len(spam), random_state=42)

balanced_df = pd.concat([spam, ham]).sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(balanced_df['label'].value_counts())

# now split: 80% to teach the model, 20% to test it
X_train, X_test, y_train, y_test = train_test_split(
    balanced_df['clean_message'], balanced_df['label'],
    test_size=0.2, random_state=42, stratify=balanced_df['label']
)

print('Training messages:', len(X_train))
print('Testing messages:', len(X_test))

label
ham     747
spam    747
Name: count, dtype: int64
Training messages: 1195
Testing messages: 299


In [8]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)   # learn vocabulary from training messages + convert them
X_test_vec = vectorizer.transform(X_test)          # convert test messages using the SAME vocabulary

print('Vocabulary size:', len(vectorizer.get_feature_names_out()))
print('X_train_vec shape:', X_train_vec.shape)
print('X_test_vec shape:', X_test_vec.shape)

Vocabulary size: 4108
X_train_vec shape: (1195, 4108)
X_test_vec shape: (299, 4108)


In [9]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_vec, y_train)

print('Model trained!')

Model trained!


In [10]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_vec)

print('Accuracy:', accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.9431438127090301

              precision    recall  f1-score   support

         ham       0.94      0.95      0.94       150
        spam       0.95      0.94      0.94       149

    accuracy                           0.94       299
   macro avg       0.94      0.94      0.94       299
weighted avg       0.94      0.94      0.94       299



In [11]:
def predict_message(text):
    cleaned = clean_text(text)              # same cleaning we did in Step 2
    vec = vectorizer.transform([cleaned])    # same word-to-number conversion as Step 4
    prediction = model.predict(vec)
    return prediction[0]

# try it out
print(predict_message("Congratulations! You won a free iPhone, click here to claim now"))
print(predict_message("Hey, are we still meeting for lunch tomorrow?"))

spam
ham


In [12]:
import pickle

with open('spam_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print('Model and vectorizer saved!')

Model and vectorizer saved!
